# NFL Roster Construction & Cap Efficiency
### Data Collection: 2013-2025

Pulls player stats, rosters, and contracts, merges them with the same
leakage-safe logic used in the NFL Player Value Engine, then aggregates
to team + position group + season for cap allocation analysis.

In [19]:
import requests

for year in range(2013, 2026):
    url = f"https://github.com/nflverse/nflverse-data/releases/download/player_stats/player_stats_{year}.parquet"
    r = requests.head(url)
    print(year, r.status_code)

2013 302
2014 302
2015 302
2016 302
2017 302
2018 302
2019 302
2020 302
2021 302
2022 302
2023 302
2024 302
2025 404


In [20]:
YEARS = list(range(2013, 2025))  # 2013-2024, confirmed working

In [21]:
import nfl_data_py as nfl
import pandas as pd

YEARS = list(range(2013, 2025))  # 2013-2024, confirmed working — 2025 not yet published

print("Pulling player stats...")
stats = nfl.import_seasonal_data(years=YEARS)
print(f"Stats shape: {stats.shape}")

Pulling player stats...
Stats shape: (7269, 58)


In [22]:
print("Pulling roster data...")
rosters = nfl.import_seasonal_rosters(years=YEARS)
print(f"Rosters shape: {rosters.shape}")

Pulling roster data...
Rosters shape: (34338, 37)


In [23]:
# Keep only the columns needed from rosters
roster_clean = rosters[['player_id', 'season', 'player_name', 'position',
                          'age', 'years_exp', 'draft_number', 'entry_year',
                          'team', 'weight', 'height']].copy()

roster_clean = roster_clean.drop_duplicates(subset=['player_id', 'season'])

df = pd.merge(stats, roster_clean, on=['player_id', 'season'], how='inner')
print(f"Merged stats+roster shape: {df.shape}")
print(f"Season range: {df['season'].min()} - {df['season'].max()}")

Merged stats+roster shape: (7269, 67)
Season range: 2013 - 2024


In [24]:
print("Pulling contract data...")
contracts = nfl.import_contracts()
print(f"Contracts shape: {contracts.shape}")

contracts_clean = contracts[['player', 'position', 'team', 'year_signed',
                               'years', 'value', 'apy', 'guaranteed',
                               'draft_overall']].copy()

contracts_clean = contracts_clean.rename(columns={'player': 'player_name', 'apy': 'aav'})

# Drop rows with missing/placeholder year_signed (e.g. 0) before merging
contracts_clean = contracts_clean[contracts_clean['year_signed'] > 2000]
print(f"Contracts after cleaning: {contracts_clean.shape}")

Pulling contract data...
Contracts shape: (51858, 25)
Contracts after cleaning: (50541, 9)


### Leakage-safe merge

Same approach as the Player Value Engine: only match a contract to a season
if it was signed on or before that season, and keep the most recent
qualifying contract per player per season.

In [25]:
master_df = pd.merge(df, contracts_clean, on=['player_name', 'position'], how='inner')
master_df = master_df[master_df['year_signed'] <= master_df['season']]
master_df = master_df.sort_values('year_signed', ascending=False)
master_df = master_df.drop_duplicates(subset=['player_name', 'season'], keep='first')

print(f"Master dataset shape: {master_df.shape}")
print(f"Season range: {master_df['season'].min()} - {master_df['season'].max()}")

Master dataset shape: (6143, 74)
Season range: 2013 - 2024


### Position group mapping

Rolling individual positions up to standard front-office position groups
so cap allocation can be compared at a meaningful level (e.g. all offensive
linemen together, not split by T/G/C).

In [26]:
position_group_map = {
    'QB': 'QB',
    'RB': 'RB', 'FB': 'RB',
    'WR': 'WR',
    'TE': 'TE',
    'T': 'OL', 'G': 'OL', 'C': 'OL', 'OL': 'OL', 'OT': 'OL', 'OG': 'OL',
    'DE': 'DL', 'DT': 'DL', 'NT': 'DL', 'DL': 'DL',
    'LB': 'LB', 'OLB': 'LB', 'ILB': 'LB', 'MLB': 'LB',
    'CB': 'CB', 'DB': 'CB',
    'S': 'S', 'FS': 'S', 'SS': 'S',
    'K': 'ST', 'P': 'ST', 'LS': 'ST',
}

master_df['position_group'] = master_df['position'].map(position_group_map)

unmapped = master_df[master_df['position_group'].isna()]['position'].unique()
print(f"Unmapped positions (check these): {unmapped}")

Unmapped positions (check these): []


In [27]:
# Team-level cap allocation by position group and season
team_cap_allocation = (
    master_df.groupby(['team_x', 'season', 'position_group'])['aav']
    .sum()
    .reset_index()
    .rename(columns={'team_x': 'team', 'aav': 'total_cap_allocated'})
)

print(f"Team cap allocation shape: {team_cap_allocation.shape}")
team_cap_allocation.head(10)

Team cap allocation shape: (1684, 4)


,team,season,position_group,total_cap_allocated
0,ARI,2016,QB,24.250000
1,ARI,2016,RB,3.301091
2,ARI,2016,ST,0.675000
3,ARI,2016,TE,6.095825
4,ARI,2016,WR,13.822175
5,ARI,2017,QB,25.105000
6,ARI,2017,RB,7.065618
7,ARI,2017,TE,8.745625
8,ARI,2017,WR,14.787247
9,ARI,2018,QB,28.399440


### Team performance data

Pulling game results to compute wins and point differential per team per
season — this becomes the outcome variable for the efficiency analysis.

In [28]:
print("Pulling schedule/results data...")
schedules = nfl.import_schedules(years=YEARS)
print(f"Schedules shape: {schedules.shape}")
schedules[['season', 'week', 'home_team', 'away_team', 'home_score', 'away_score']].head()

Pulling schedule/results data...
Schedules shape: (3277, 46)


,season,week,home_team,away_team,home_score,away_score
3714,2013,1,DEN,BAL,49.0,27.0
3715,2013,1,BUF,NE,21.0,23.0
3716,2013,1,CAR,SEA,7.0,12.0
3717,2013,1,CHI,CIN,24.0,21.0
3718,2013,1,CLE,MIA,10.0,23.0


In [29]:
# Build a team-season win/point-differential table from schedule results
completed = schedules.dropna(subset=['home_score', 'away_score']).copy()

home = completed[['season', 'home_team', 'home_score', 'away_score']].rename(
    columns={'home_team': 'team', 'home_score': 'points_for', 'away_score': 'points_against'})
away = completed[['season', 'away_team', 'away_score', 'home_score']].rename(
    columns={'away_team': 'team', 'away_score': 'points_for', 'home_score': 'points_against'})

team_games = pd.concat([home, away], ignore_index=True)
team_games['win'] = (team_games['points_for'] > team_games['points_against']).astype(int)

team_performance = team_games.groupby(['team', 'season']).agg(
    wins=('win', 'sum'),
    games=('win', 'count'),
    point_diff=('points_for', lambda x: x.sum() - team_games.loc[x.index, 'points_against'].sum())
).reset_index()

print(f"Team performance shape: {team_performance.shape}")
team_performance.head(10)

Team performance shape: (384, 5)


,team,season,wins,games,point_diff
0,ARI,2013,10,16,55.0
1,ARI,2014,11,17,0.0
2,ARI,2015,14,18,148.0
3,ARI,2016,7,16,56.0
4,ARI,2017,8,16,-66.0
5,ARI,2018,3,16,-200.0
6,ARI,2019,5,16,-81.0
7,ARI,2020,8,16,43.0
8,ARI,2021,11,18,60.0
9,ARI,2022,4,17,-109.0


In [30]:
import os
os.makedirs('../data', exist_ok=True)

In [31]:
# Save intermediate outputs to data folder
team_cap_allocation.to_csv('../data/team_cap_allocation.csv', index=False)
team_performance.to_csv('../data/team_performance.csv', index=False)

print("Saved team_cap_allocation.csv and team_performance.csv")

Saved team_cap_allocation.csv and team_performance.csv


In [32]:
team_cap_allocation.head(10)
team_performance.head(10)

,team,season,wins,games,point_diff
0,ARI,2013,10,16,55.0
1,ARI,2014,11,17,0.0
2,ARI,2015,14,18,148.0
3,ARI,2016,7,16,56.0
4,ARI,2017,8,16,-66.0
5,ARI,2018,3,16,-200.0
6,ARI,2019,5,16,-81.0
7,ARI,2020,8,16,43.0
8,ARI,2021,11,18,60.0
9,ARI,2022,4,17,-109.0
